# NBA Talent Intelligence: Early-Career Talent Screening Prototype

**Purpose.** Test whether first- and second-season box-score statistics and available player attributes can help prioritise players for further scouting, using later All-Star appearance as a retrospective proxy for future success.

**Decision supported.** Which early-career players should receive additional scouting/development attention? This prototype is decision support only; it does not make recruitment, contract or player-development decisions.

**Important limitations.**
- The outcome is a proxy (All-Star appearance in career seasons 3–7), not a complete definition of player value.
- Only players without an All-Star appearance in their first two observed regular seasons are included.
- The data is historical and box-score based; it does not include contract, medical, tracking, private scouting or player-development assessment data.
- A temporal holdout is used, but historical rule, pace and role differences remain.
- Report results as retrospective evidence, not proof of future operational impact.

## Setup
Place `PlayerStatistics.csv` and `Players.csv` beside this notebook. The code also recognises the uploaded names `PlayerStatistics(1).csv` and `Players(1).csv`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def locate_file(candidates):
    for name in candidates:
        p = Path(name)
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find any of: {candidates}. Put the source CSVs beside this notebook.")

stats_path = locate_file(["PlayerStatistics.csv", "PlayerStatistics(1).csv"])
players_path = locate_file(["Players.csv", "Players(1).csv"])
print("Statistics:", stats_path, f"({stats_path.stat().st_size/1e6:.1f} MB)")
print("Players:", players_path, f"({players_path.stat().st_size/1e6:.2f} MB)")


## 1. Load and inspect the source data

In [ ]:
stats = pd.read_csv(stats_path, low_memory=False)
players = pd.read_csv(players_path, low_memory=False)

print("PlayerStatistics shape:", stats.shape)
print("Players shape:", players.shape)
print("\nGame types:")
display(stats["gameType"].value_counts(dropna=False).head(15).to_frame("rows"))
print("\nPlayer ID uniqueness:")
print("Players rows:", len(players), "| unique personId:", players["personId"].nunique(),
      "| duplicate IDs:", int(players["personId"].duplicated().sum()))
print("Statistics date range:", pd.to_datetime(stats["gameDate"], errors="coerce").min(),
      "to", pd.to_datetime(stats["gameDate"], errors="coerce").max())


## 2. Prepare season-level data

The NBA season is labelled by its starting calendar year: October–December games use that year, while January–June games use the preceding year. Only rows labelled `Regular Season` are used for early-career performance features. All-Star Game rows are used only to define the retrospective target.


In [ ]:
stats["gameDate"] = pd.to_datetime(stats["gameDate"], errors="coerce")
stats["season"] = stats["gameDate"].dt.year - (stats["gameDate"].dt.month < 10).astype("Int64")
regular = stats[
    stats["gameType"].astype(str).str.strip().str.lower().eq("regular season")
    & stats["gameDate"].notna()
].copy()
regular["season"] = regular["season"].astype(int)

numeric = [
    "numMinutes", "points", "assists", "blocks", "steals", "fieldGoalsAttempted",
    "fieldGoalsMade", "fieldGoalsPercentage", "threePointersAttempted",
    "threePointersMade", "threePointersPercentage", "freeThrowsAttempted",
    "freeThrowsMade", "freeThrowsPercentage", "reboundsDefensive",
    "reboundsOffensive", "reboundsTotal", "foulsPersonal", "turnovers", "plusMinusPoints"
]
for col in numeric:
    regular[col] = pd.to_numeric(regular[col], errors="coerce")

season_stats = regular.groupby(["personId", "season"]).agg(
    games=("gameId", "nunique"),
    minutes_pg=("numMinutes", "mean"),
    points_pg=("points", "mean"),
    assists_pg=("assists", "mean"),
    blocks_pg=("blocks", "mean"),
    steals_pg=("steals", "mean"),
    rebounds_pg=("reboundsTotal", "mean"),
    turnovers_pg=("turnovers", "mean"),
    plusminus_pg=("plusMinusPoints", "mean"),
    fg_pct=("fieldGoalsPercentage", "mean"),
    three_pct=("threePointersPercentage", "mean"),
    ft_pct=("freeThrowsPercentage", "mean"),
    fga_pg=("fieldGoalsAttempted", "mean"),
    tpa_pg=("threePointersAttempted", "mean"),
    fta_pg=("freeThrowsAttempted", "mean")
).reset_index().sort_values(["personId", "season"])
season_stats["career_season_num"] = season_stats.groupby("personId").cumcount() + 1

allstar = stats[
    stats["gameType"].astype(str).str.strip().str.lower().str.contains("all-star", na=False)
    & stats["gameDate"].notna()
].copy()
allstar["season"] = (
    allstar["gameDate"].dt.year - (allstar["gameDate"].dt.month < 10).astype(int)
).astype(int)
allstar_seasons = allstar[["personId", "season"]].drop_duplicates()
allstar_seasons["allstar"] = 1
season_stats = season_stats.merge(allstar_seasons, on=["personId", "season"], how="left")
season_stats["allstar"] = season_stats["allstar"].fillna(0).astype(int)

print("Regular-season player-season rows:", len(season_stats))
print("Season range:", season_stats["season"].min(), "to", season_stats["season"].max())
print("Players with regular-season data:", season_stats["personId"].nunique())


## 3. Define the outcome and construct early-career features

In [ ]:
feature_cols = [
    "games", "minutes_pg", "points_pg", "assists_pg", "blocks_pg", "steals_pg",
    "rebounds_pg", "turnovers_pg", "plusminus_pg", "fg_pct", "three_pct", "ft_pct",
    "fga_pg", "tpa_pg", "fta_pg"
]
season1 = season_stats[season_stats["career_season_num"] == 1].set_index("personId")
season2 = season_stats[season_stats["career_season_num"] == 2].set_index("personId")
common_ids = season1.index.intersection(season2.index)

early = pd.DataFrame({"personId": common_ids})
for col in feature_cols:
    early[f"s1_{col}"] = season1.loc[common_ids, col].to_numpy()
    early[f"s2_{col}"] = season2.loc[common_ids, col].to_numpy()
early["debut_season"] = season1.loc[common_ids, "season"].to_numpy()
early["allstar_s1_s2"] = (
    season1.loc[common_ids, "allstar"].to_numpy()
    + season2.loc[common_ids, "allstar"].to_numpy()
)

future_allstar = (
    season_stats[season_stats["career_season_num"].between(3, 7)]
    .groupby("personId")["allstar"].max()
)
early["future_allstar_3to7"] = early["personId"].map(future_allstar).fillna(0).astype(int)

players["birthDate"] = pd.to_datetime(players["birthDate"], errors="coerce")
player_meta = players.set_index("personId")
meta_features = [
    "heightInches", "bodyWeightLbs", "guard", "forward", "center",
    "draftYear", "draftRound", "draftNumber"
]
for col in meta_features:
    early[col] = early["personId"].map(player_meta[col])
birth_year = early["personId"].map(player_meta["birthDate"].dt.year)
early["age_at_debut"] = early["debut_season"] - birth_year
early["player_name"] = early["personId"].map(
    (players["firstName"].fillna("") + " " + players["lastName"].fillna(""))
    .str.strip().set_axis(players["personId"])
)

latest_season = int(regular["season"].max())
eligible = early[
    (early["debut_season"] + 6 <= latest_season)
    & (early["allstar_s1_s2"] == 0)
].copy()
eligible["future_allstar_3to7"] = eligible["future_allstar_3to7"].astype(int)

print("Eligible players:", len(eligible))
print("Positive future All-Star outcomes:", int(eligible["future_allstar_3to7"].sum()))
print("Outcome prevalence:", round(eligible["future_allstar_3to7"].mean() * 100, 2), "%")
display(eligible[["player_name", "debut_season", "s1_points_pg", "s2_points_pg",
                  "future_allstar_3to7"]].head())


## 4. Temporal train/test split and baseline

Training uses players whose first observed regular season began through 2010. Testing uses debut seasons 2011–2019, allowing seasons 3–7 to be observed in the available data. The baseline ranks players by mean points per game across their first two seasons. This is a deliberately simple benchmark, not a claim that every franchise currently uses this exact process.


In [ ]:
model_features = [
    f"s{n}_{col}" for n in (1, 2) for col in feature_cols
] + ["age_at_debut", "heightInches", "bodyWeightLbs", "guard", "forward", "center",
     "draftYear", "draftRound", "draftNumber"]

train = eligible[eligible["debut_season"] <= 2010].copy()
test = eligible[eligible["debut_season"].between(2011, 2019)].copy()
X_train = train[model_features].replace([np.inf, -np.inf], np.nan)
y_train = train["future_allstar_3to7"]
X_test = test[model_features].replace([np.inf, -np.inf], np.nan)
y_test = test["future_allstar_3to7"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1, max_features="sqrt"
    ))
])
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]

k = max(1, int(np.ceil(len(test) * 0.10)))
model_order = np.argsort(-probabilities)
baseline_score = (
    test["s1_points_pg"].fillna(0).to_numpy()
    + test["s2_points_pg"].fillna(0).to_numpy()
) / 2
baseline_order = np.argsort(-baseline_score)

results = {
    "model": {
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average precision": average_precision_score(y_test, probabilities),
        "Brier score": brier_score_loss(y_test, probabilities),
        "Precision@10%": y_test.iloc[model_order[:k]].mean(),
        "Recall@10%": y_test.iloc[model_order[:k]].sum() / max(1, y_test.sum())
    },
    "baseline": {
        "ROC-AUC": roc_auc_score(y_test, baseline_score),
        "Average precision": average_precision_score(y_test, baseline_score),
        "Precision@10%": y_test.iloc[baseline_order[:k]].mean(),
        "Recall@10%": y_test.iloc[baseline_order[:k]].sum() / max(1, y_test.sum())
    }
}
print(f"Train: {len(train):,} players ({int(y_train.sum())} positive outcomes)")
print(f"Temporal test: {len(test):,} players ({int(y_test.sum())} positive outcomes)")
print(f"Top 10% shortlist size K = {k}")
display(pd.DataFrame(results).round(3))


## 5. Evaluate shortlist quality

Precision@K answers: of the K players ranked highest, what fraction later appeared in an All-Star Game during career seasons 3–7? Recall@K answers: what fraction of all positive cases in the test set appeared in the top-K shortlist. ROC-AUC evaluates ranking discrimination; average precision is useful for an imbalanced target; Brier score evaluates probability error (lower is better).

### Initial retrospective result

The supplied data produced 3,162 eligible players, with 2,501 in training and 661 in the temporal test set. The Random Forest achieved ROC-AUC **0.949**, average precision **0.558**, Brier score **0.037**, and Precision@10% **0.373**. The mean-points-per-game baseline achieved ROC-AUC **0.905**, average precision **0.445**, and Precision@10% **0.328**.

These are promising retrospective results, not proof of business impact. The holdout contains 36 positive outcomes, so uncertainty is material. The target is All-Star selection, which is affected by popularity, role, team success, conference, voting and selection rules—not solely player quality. Further temporal and era-aware robustness checks are required before operational use.


In [ ]:
comparison = pd.DataFrame({
    "Approach": ["Mean PPG baseline", "Random Forest"],
    "ROC-AUC": [results["baseline"]["ROC-AUC"], results["model"]["ROC-AUC"]],
    "Average precision": [results["baseline"]["Average precision"], results["model"]["Average precision"]],
    "Precision@10%": [results["baseline"]["Precision@10%"], results["model"]["Precision@10%"]],
    "Recall@10%": [results["baseline"]["Recall@10%"], results["model"]["Recall@10%"]]
})
ax = comparison.set_index("Approach")[["Precision@10%"]].plot(kind="bar", legend=False, figsize=(7,4))
ax.set_ylabel("Precision in top 10% shortlist")
ax.set_title("Shortlist precision: temporal holdout")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Save scored test cohort for dashboard

The exported cohort is a retrospective evaluation view, not a live list of current unsigned players. Actual outcomes are retained for evaluation, so the dashboard must label this view as historical and must not be used to make current recruitment decisions.


In [ ]:
scored = test.copy()
scored["predicted_probability"] = probabilities
scored["baseline_mean_points_pg"] = baseline_score
scored["actual_future_allstar"] = y_test.to_numpy()
scored["model_rank"] = pd.Series(probabilities, index=test.index).rank(method="first", ascending=False).astype(int)
scored["baseline_rank"] = pd.Series(baseline_score, index=test.index).rank(method="first", ascending=False).astype(int)
display(scored.sort_values("predicted_probability", ascending=False)[
    ["player_name", "debut_season", "predicted_probability", "model_rank",
     "baseline_rank", "actual_future_allstar"]
].head(20))
scored.to_csv("scored_test_players.csv", index=False)
with open("evaluation_metrics.json", "w") as f:
    json.dump({
        "target": "All-Star appearance in career seasons 3–7; no All-Star in seasons 1–2",
        "train_n": int(len(train)), "train_positive": int(y_train.sum()),
        "test_n": int(len(test)), "test_positive": int(y_test.sum()), "k": int(k),
        "model": {key: float(value) for key, value in results["model"].items()},
        "baseline": {key: float(value) for key, value in results["baseline"].items()}
    }, f, indent=2)
